# Desarrollo y pruebas del modelo base (Baseline-model)



In [29]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from utils import *
from models import *

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Normalization, Conv1D, Dense

In [30]:
## Levanto el dataset (unicamente los datos de acelerometro)
data = extraer_datos_csv(path='./Datasets/outputs_FallAI_Dataset', file = 'Event_FallAIIDDataSet_*.txt', sensors= ['AccX','AccY','AccZ','Bar','Temp'])

## Extraigo los Tags del dataset
tags = extraer_tags_csv(path='./Datasets/outputs_FallAI_Dataset', file = 'Event_FallAIIDDataSet_*.txt')


In [31]:
data_entrenamiento, data_val, tags_entrenamiento, y_val  = train_test_split(data,tags, test_size=0.1, random_state=21)

# data_train_presion, data_test_presion, y_train_presion, y_test_presion = train_test_split(data_sensores,tags_presion, test_size=0.2, random_state=21)

In [32]:
data_presion = data_val[:,:,3:5]
alturas_mediciones = calcular_altura(data_presion,20)
features_extracted_presion = extraer_features(alturas_mediciones) # inputs modelo de sensor de presión

data_acelerometro = data_val[:,:,:3]
escalador = dataScaler()
data_acelerometro = escalador.escalar_datos(data_acelerometro)


In [33]:
modelo_sensor_presion= tf.keras.models.load_model("./Modelos/modelo_barometro.keras")

modelo_sensor_acelerometro= tf.keras.models.load_model("./Modelos/modelo_acelerometro.keras")

In [38]:
y_pred_presion = modelo_sensor_presion.predict(features_extracted_presion)
y_pred_acelerometro = modelo_sensor_acelerometro.predict(data_acelerometro)

y_pred = (y_pred_presion * 0.30 + y_pred_acelerometro * 0.70) 

threshold = 0.232
y_pred = (y_pred > threshold ).astype(int)
print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_val, y_pred, target_names=["NOT FALL (0)", "FALL (1)"]))


matriz = confusion_matrix(y_val, y_pred)
print("\n=== MATRIZ DE CONFUSIÓN ===")
print("                  Predice NO CAÍDA   Predice CAÍDA")
print(f"Real NO CAÍDA:    {matriz[0][0]}                 {matriz[0][1]}")
print(f"Real CAÍDA:       {matriz[1][0]}                 {matriz[1][1]}")

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
=== REPORTE DE CLASIFICACIÓN ===
              precision    recall  f1-score   support

NOT FALL (0)       1.00      0.92      0.96       205
    FALL (1)       0.73      1.00      0.85        47

    accuracy                           0.93       252
   macro avg       0.87      0.96      0.90       252
weighted avg       0.95      0.93      0.94       252


=== MATRIZ DE CONFUSIÓN ===
                  Predice NO CAÍDA   Predice CAÍDA
Real NO CAÍDA:    188                 17
Real CAÍDA:       0                 47
